# Functions to calculate ribosomal density and number of ribosomes.


In [1]:
import sys; from pathlib import Path
src_dir = next(parent / 'src' for parent in Path().absolute().parents if (parent / 'src').is_dir())
sys.path.extend([str(src_dir), str(src_dir / 'pipelines')])
main_dir = Path(src_dir.parents[0])
from imports import * 
current_dir = Path().resolve() 

In [2]:
# Modeling
from tasep_models import *

In [3]:
# Function to extract ki and ke from param files
def extract_params(filename):
    df = pd.read_csv(results_folder / filename)
    ki_value = df[df['parameter'] == 'ki']['value'].iloc[0]
    ke_value = df[df['parameter'] == 'ke']['value'].iloc[0]
    return float(ke_value), float(ki_value)

In [4]:
def model_and_data_selection(dataset):
    if dataset == 'utag':
        tag_sequence = tag_dict['U']
        dna_file_path = main_dir.joinpath( 'gene_sequences','utag_project','pNZ208(pUB-24xUTagFullLength-KDM5B-MS2).dna' )
        ke_utag, ki_utag = extract_params('optimized_params_utag.csv')
        list_param = [ki_utag, ke_utag, 'utag']
    elif dataset == 'utag_c_free':
        tag_sequence = tag_dict['U']
        dna_file_path = main_dir.joinpath( 'gene_sequences','utag_project','pNZ208(pUB-24xUTagFullLength-KDM5B-MS2).dna' )
        ke_utag_c_free, ki_utag_c_free = extract_params('optimized_params_utag_c_free.csv')
        list_param = [ki_utag_c_free, ke_utag_c_free, 'utag_c_free']
    elif dataset == 'suntag':
        tag_sequence = tag_dict['SUN']
        dna_file_path = main_dir.joinpath( 'gene_sequences','utag_project','pNZ266(pUB-24xGCN4-KDM5B-MS2).dna' )
        ke_suntag, ki_suntag = extract_params('optimized_params_suntag.csv')
        list_param = [ki_suntag, ke_suntag, 'suntag']
    elif dataset == 'alfatag':
        tag_sequence = tag_dict['ALFA']
        dna_file_path = main_dir.joinpath( 'gene_sequences','utag_project','pNZ267 (pUB-24xALFAtag-KDM5B-MS2).dna' )
        ke_alfatag, ki_alfatag = extract_params('optimized_params_alfatag.csv')
        list_param = [ki_alfatag, ke_alfatag, 'alfatag']
    # reading the gene sequence
    protein, rna, dna, indexes_tags, _, seq_record, graphic_features  = read_sequence(seq=dna_file_path, min_protein_length=50,TAG=[tag_sequence])
    gene_length = len(protein)+1 # adding 1 to account for the stop codon
    tag_positions_first_probe_vector = indexes_tags[0]
    first_probe_position_vector = create_probe_vector(tag_positions_first_probe_vector, gene_length)
    return list_param[2], rna, first_probe_position_vector, gene_length, list_param

In [5]:
# process and print all datasets

list_datasets = ['utag','utag_c_free','suntag','alfatag']
#results_folder = Path('/Users/nzlab-la/Desktop/utag/optimization/results_ACF')

results_folder = Path('/Users/nzlab-la/Desktop/utag_paper/optimization/results_ACF')
for i, dataset in enumerate(list_datasets):
    model_name, rna, first_probe_position_vector, gene_length, list_param = model_and_data_selection(dataset)
    ki, ke = list_param[0], list_param[1]
    #print(f"Dataset: {dataset}, ki: {np.round(ki,3)}, ke: {np.round(ke,3)}, gene_length: {gene_length}")
    ki_rounded = np.round(ki, 3)
    ke_rounded = np.round(ke, 2)
    print(f"Tag: {dataset:<15} ki: {ki_rounded:<8} ke: {ke_rounded:<8} gene_length: {gene_length:<5}")


Tag: utag            ki: 0.031    ke: 3.11     gene_length: 1968 
Tag: utag_c_free     ki: 0.063    ke: 5.29     gene_length: 1968 
Tag: suntag          ki: 0.034    ke: 4.33     gene_length: 2133 
Tag: alfatag         ki: 0.043    ke: 4.84     gene_length: 2016 


In [6]:
# ribosomal density calculation
def calculate_ribosomal_density(ki, ke, ribosomal_footprint=10):
    ribosomal_density= (ki * ribosomal_footprint) / ke
    ribosomal_density = ribosomal_density * 100
    return np.round(ribosomal_density, 3)

def calculate_number_ribosomes_RNA(ki, ke, gene_length):
    number_ribosomes = (ki * gene_length) / ke
    return np.round(number_ribosomes, 3)

def calculate_ribosomal_distance(gene_length, number_ribosomes):
    ribosomal_distance = gene_length / number_ribosomes
    return np.round(ribosomal_distance, 3)


In [8]:
# process and print all datasets with ribosomal density and number of ribosomes per RNA.
list_datasets = ['utag','utag_c_free','suntag','alfatag']
#results_folder = Path('/Users/nzlab-la/Desktop/utag/optimization/results_ACF')
for i, dataset in enumerate(list_datasets):
    model_name, rna, first_probe_position_vector, gene_length, list_param = model_and_data_selection(dataset)
    ki, ke = list_param[0], list_param[1]
    ki_rounded = np.round(ki, 3)
    ke_rounded = np.round(ke, 2)
    ribosomal_density = np.round(calculate_ribosomal_density(ki, ke, ribosomal_footprint=9),1)
    number_ribosomes = np.round(calculate_number_ribosomes_RNA(ki, ke, gene_length),1)
    ribosomal_distance = np.round(calculate_ribosomal_distance(gene_length, number_ribosomes),1)
    print(f"Tag: {dataset:<15} ki: {ki_rounded:<8} ke: {ke_rounded:<8} length: {gene_length:<5} rib_density: {ribosomal_density:<8} no_rib: {number_ribosomes:<8} rib_dist: {ribosomal_distance:<8}" )

Tag: utag            ki: 0.031    ke: 3.11     length: 1968  rib_density: 9.0      no_rib: 19.8     rib_dist: 99.4    
Tag: utag_c_free     ki: 0.063    ke: 5.29     length: 1968  rib_density: 10.7     no_rib: 23.5     rib_dist: 83.7    
Tag: suntag          ki: 0.034    ke: 4.33     length: 2133  rib_density: 7.1      no_rib: 16.7     rib_dist: 127.7   
Tag: alfatag         ki: 0.043    ke: 4.84     length: 2016  rib_density: 8.0      no_rib: 17.8     rib_dist: 113.3   
